In [6]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = pd.read_csv("qoute_dataset.csv")
# Location of the dataset should be mentioned correctly

In [8]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [9]:
df.shape

(3038, 2)

# NLP Step

In [11]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [12]:
quotes = quotes.str.lower()

In [14]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [15]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [16]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [17]:
vocab_size = 10000

tokinizer = Tokenizer(num_words=vocab_size)
tokinizer.fit_on_texts(quotes)

In [18]:
word_index = tokinizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [19]:
sequence = tokinizer.texts_to_sequences(quotes)

In [20]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [21]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [22]:
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [23]:
len(X)

85271

In [24]:
len(y)

85271

In [25]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [26]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [27]:
y = np.array(y)

In [28]:
X_padded.shape

(85271, 745)

In [29]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [30]:
y.shape

(85271,)

In [31]:
y_one_hot.shape

(85271, 10000)

In [32]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense,GRU

In [33]:
embedding_dim = 50
rnn_units = 128

# RNN Model

In [34]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

In [35]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [36]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# LSTM Model

In [37]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [38]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [39]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [44]:
batch_size = 128
epochs = 100

In [35]:
history_rnn = rnn_model.fit(
    X_padded, y_one_hot,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 52s 78ms/step - accuracy: 0.0431 - loss: 6.7779 - val_accuracy: 0.0556 - val_loss: 6.5903
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 69ms/step - accuracy: 0.0697 - loss: 6.1935 - val_accuracy: 0.0860 - val_loss: 6.3975
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 42s 70ms/step - accuracy: 0.0995 - loss: 5.8342 - val_accuracy: 0.1011 - val_loss: 6.2856
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 69ms/step - accuracy: 0.1166 - loss: 5.5323 - val_accuracy: 0.1081 - val_loss: 6.2831
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.1291 - loss: 5.2651 - val_accuracy: 0.1103 - val_loss: 6.3317
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 69ms/step - accuracy: 0.1436 - loss: 5.0212 - val_accuracy: 0.1079 - val_loss: 6.3866
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.1579 - loss: 4.7941 - val_accuracy: 0.1080 - val_loss: 6.4763
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.1743 - loss: 4

In [36]:
epochs= 100
batch_size =128
history_lstm= lstm_model.fit(
    X_padded, y_one_hot,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 52ms/step - accuracy: 0.0397 - loss: 6.7549 - val_accuracy: 0.0492 - val_loss: 6.6850
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.0597 - loss: 6.3060 - val_accuracy: 0.0687 - val_loss: 6.5236
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.0837 - loss: 6.0312 - val_accuracy: 0.0908 - val_loss: 6.4468
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.1008 - loss: 5.8127 - val_accuracy: 0.0972 - val_loss: 6.4026
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1128 - loss: 5.6215 - val_accuracy: 0.1039 - val_loss: 6.3810
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1223 - loss: 5.4482 - val_accuracy: 0.1073 - val_loss: 6.3935
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1304 - loss: 5.2922 - val_accuracy: 0.1079 - val_loss: 6.4221
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1389 - loss: 5

# GRU Model

In [40]:
gru_model = Sequential()
gru_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
gru_model.add(GRU(units=rnn_units))
gru_model.add(Dense(units=vocab_size, activation='softmax'))

In [41]:
gru_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [42]:
gru_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [45]:
epochs= 100
batch_size =128
gru_model.fit(
    X_padded, y_one_hot,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 36s 49ms/step - accuracy: 0.0422 - loss: 6.7872 - val_accuracy: 0.0611 - val_loss: 6.5403
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 29s 48ms/step - accuracy: 0.0805 - loss: 6.0860 - val_accuracy: 0.0923 - val_loss: 6.3256
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 29s 48ms/step - accuracy: 0.1041 - loss: 5.7348 - val_accuracy: 0.1042 - val_loss: 6.2676
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 29s 48ms/step - accuracy: 0.1203 - loss: 5.4395 - val_accuracy: 0.1123 - val_loss: 6.2885
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 29s 48ms/step - accuracy: 0.1381 - loss: 5.1743 - val_accuracy: 0.1174 - val_loss: 6.3160
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 29s 49ms/step - accuracy: 0.1517 - loss: 4.9297 - val_accuracy: 0.1188 - val_loss: 6.3543
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 29s 49ms/step - accuracy: 0.1637 - loss: 4.7046 - val_accuracy: 0.1215 - val_loss: 6.4392
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 29s 49ms/step - accuracy: 0.1791 - loss: 4

In [46]:
lstm_model.save("lstm_model.h5")
rnn_model.save("rnn_model.h5")
gru_model.save("gru_model.h5")

In [60]:
from tensorflow.keras.models import load_model
lstm_model = load_model("lstm_model.h5")
rnn_model = load_model("rnn_model.h5")
gru_model = load_model("gru_model.h5")

In [48]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [49]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [50]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [51]:
seed_text = "what are you"
next_word = predictor(lstm_model,tokinizer,seed_text,max_len)
print(next_word)

thought


In [52]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [55]:
seed = "are you a "
generate_text = generate_text(lstm_model,tokinizer,seed,max_len,10)
print(generate_text)

are you a  shelf beat beat spans spans filthy filthy redeemednever professions fastly


In [65]:
seed_text = "what are you"

# RNN prediction
rnn_prediction = predictor(
    rnn_model, tokinizer, seed_text, max_len
)

# LSTM prediction
lstm_prediction = predictor(
    lstm_model, tokinizer, seed_text, max_len
)

# GRU prediction
gru_prediction = predictor(
    gru_model, tokinizer, seed_text, max_len
)

print("Input Text:", seed_text)
print("-" * 35)
print("RNN Prediction:", rnn_prediction)
print("LSTM Prediction:", lstm_prediction)
print("GRU Prediction:", gru_prediction)

Input Text: what are you
-----------------------------------
RNN Prediction: nap
LSTM Prediction: whose
GRU Prediction: know


In [66]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokinizer, f)

In [67]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)